Install LangGraph

In [1]:
%pip install -U langgraph

  Attempting uninstall: langgraph
    Found existing installation: langgraph 1.2.16
    Uninstalling langgraph-1.2.16:
      Successfully uninstalled langgraph-1.2.16
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Create the shared state

In [2]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

class AgentState(TypedDict):
    user_query: str
    research_data: str
    final_answer: str
    error: str
    communication_log: list

Create Coordinator Agent

In [3]:
def coordinator(state):
    print("\n[COORDINATOR AGENT]")
    print("User Request:", state["user_query"])
    print("Assigning task to Research Agent...")

    return {
        "communication_log":
        state["communication_log"] + [
            "Coordinator received user request",
            "Coordinator assigned task to Research Agent"
        ]
    }

Create Research Agent

In [4]:
def research_agent(state):
    print("\n[RESEARCH AGENT]")
    print("Researching:", state["user_query"])

    try:
        if "fail" in state["user_query"].lower():
            raise Exception("Research service unavailable")

        research = (
            "AI improves business productivity, "
            "automation, decision-making and "
            "customer service."
        )

        print("Research completed!")
        print("Research Result:", research)

        return {
            "research_data": research,
            "error": "",
            "communication_log":
            state["communication_log"] + [
                "Research Agent received task",
                "Research Agent collected information",
                "Research Agent sent findings"
            ]
        }

    except Exception as e:
        print("ERROR:", e)

        return {
            "research_data": "Research unavailable",
            "error": str(e),
            "communication_log":
            state["communication_log"] + [
                "Research Agent encountered an error",
                "Fallback response activated"
            ]
        }

Create Writer Agent

In [5]:
def writer_agent(state):
    print("\n[WRITER AGENT]")
    print("Generating final response...")

    if state["error"]:
        answer = (
            "Sorry, research could not be completed. "
            + state["error"]
        )
    else:
        answer = (
            "FINAL REPORT\n"
            "User Query: " + state["user_query"] +
            "\nResearch Findings: " +
            state["research_data"] +
            "\nConclusion: AI improves business efficiency."
        )

    print("Response generated!")

    return {
        "final_answer": answer,
        "communication_log":
        state["communication_log"] + [
            "Writer Agent received research data",
            "Writer Agent generated final response",
            "Workflow completed"
        ]
    }

Connect agents using LangGraph

In [6]:
workflow = StateGraph(AgentState)

workflow.add_node("coordinator", coordinator)
workflow.add_node("research", research_agent)
workflow.add_node("writer", writer_agent)

workflow.add_edge(START, "coordinator")
workflow.add_edge("coordinator", "research")
workflow.add_edge("research", "writer")
workflow.add_edge("writer", END)

app = workflow.compile()

print("Multi-Agent System Created Successfully!")

Multi-Agent System Created Successfully!


Enter a user query

In [12]:
query = input("Enter your question: ")

initial_state = {
    "user_query": query,
    "research_data": "",
    "final_answer": "",
    "error": "",
    "communication_log": []
}

result = app.invoke(initial_state)


[COORDINATOR AGENT]
User Request: about ai
Assigning task to Research Agent...

[RESEARCH AGENT]
Researching: about ai
Research completed!
Research Result: AI improves business productivity, automation, decision-making and customer service.

[WRITER AGENT]
Generating final response...
Response generated!


Display final response

In [8]:
print("FINAL ANSWER")
print(result["final_answer"])

FINAL ANSWER
FINAL REPORT
User Query: Explain the benefits of AI in business
Research Findings: AI improves business productivity, automation, decision-making and customer service.
Conclusion: AI improves business efficiency.


Display shared state

In [9]:
print("SHARED STATE")

for key, value in result.items():
    print(f"\n{key}:")
    print(value)

SHARED STATE

user_query:
Explain the benefits of AI in business

research_data:
AI improves business productivity, automation, decision-making and customer service.

final_answer:
FINAL REPORT
User Query: Explain the benefits of AI in business
Research Findings: AI improves business productivity, automation, decision-making and customer service.
Conclusion: AI improves business efficiency.

error:


communication_log:
['Coordinator received user request', 'Coordinator assigned task to Research Agent', 'Research Agent received task', 'Research Agent collected information', 'Research Agent sent findings', 'Writer Agent received research data', 'Writer Agent generated final response', 'Workflow completed']


Display communication log

In [10]:
print("AGENT COMMUNICATION LOG")

for i, message in enumerate(
    result["communication_log"], 1
):
    print(f"{i}. {message}")

AGENT COMMUNICATION LOG
1. Coordinator received user request
2. Coordinator assigned task to Research Agent
3. Research Agent received task
4. Research Agent collected information
5. Research Agent sent findings
6. Writer Agent received research data
7. Writer Agent generated final response
8. Workflow completed
